# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports checkpoints to `/kaggle/working`.

Current experiment: **E007-unified-multiplane** — one model for the whole study.
Per study: a bag of 2.5D triplets from every available plane (3 anchors x up to 3
planes) -> one shared fine-tuned DINOv2 backbone -> learned per-plane embeddings
-> per-label attention over the whole bag -> 12 logits. This retires the
hand-coded plane-prior combiner (E002: a null): per-label attention over the
mixed bag IS the learned plane weighting, and missing planes just shrink the bag.
Same fixed 90/10 holdout (seed 0) as E006 — the comparison is paired.

GPU cost, per arm: decode ~75 min (unavoidable — /tmp doesn't survive between
kernel runs) + one unified fine-tune ~45-60 min. Total ~2-2.5 T4 hours, single
arm, nothing re-paid. Requires `WANDB_API_KEY`, `knee-labels` (and
`knee-e006-artifacts` if warm-starting), and internet.

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
# --no-deps everywhere: Kaggle's image already ships torch/timm/sklearn/numpy compiled
# together; letting pip resolve our pins upgrades numpy and breaks the whole stack.
# STALE for E007 — bump to the multiplane squash merge before `kaggle kernels push`
# (this run needs MultiPlaneModel/finetune_unified, absent at 997f2df).
COMMIT = "997f2df"
%pip install -q --no-deps "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee"
%pip install -q --no-deps pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg pylibjpeg-rle

import numpy  # fail fast if the image stack is broken or missing
import timm
import torch

print("numpy", numpy.__version__, "| torch", torch.__version__, "| timm", timm.__version__)

from knee.series import SeriesType
from knee.train_blended import BLENDED_LABEL_SOURCE

In [ ]:
# Competition data (DICOMs + series metadata) is pre-mounted; blended soft labels
# (with the per-cell __weight companions for E006a) via the knee-labels dataset;
# the E005 winner's feature bank + checkpoints via knee-e005-artifacts.
from pathlib import Path

from knee.data import load_blended_labels, weight_matrix

SLUG = "rsna-knee-abnormality-detection"
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP_ROOT = next((p for p in candidates if (p / "train.csv").exists()), None)
if COMP_ROOT is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"competition data not found; mounts: {listing}")
print("competition root:", COMP_ROOT)


def find_input(name: str, filename: str) -> Path:
    bases = [Path("/kaggle/input") / name, Path("/kaggle/input/datasets/josiemachalek") / name]
    for base in bases:
        if (base / filename).exists():
            return base / filename
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"{name}/{filename} not found; mounts: {listing}")


labels = load_blended_labels(find_input("knee-labels", "blended_labels_v1.csv"), include_weights=True)
weights = weight_matrix(labels)
print(f"blended labels: {len(labels)} studies; weights {weights.shape}")

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
# Best-effort: the run proceeds without wandb if the secret isn't configured.
run = None
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    run = wandb.init(
        project="rsna-knee",
        config={"commit": COMMIT, "label_source": BLENDED_LABEL_SOURCE, "n_label_studies": len(labels)},
    )
except Exception as exc:  # noqa: BLE001 — telemetry must never kill a training run
    print(f"wandb disabled: {exc}")

In [ ]:
# E007 config — same geometry/labels/loss as E006 (paired comparison); the one
# placeholder is the warm start, filled from E006's result: its best per-plane
# checkpoint filename (backbone + attention head copy over; plane embeddings start
# fresh), or None for pretrained DINOv2 + cold head. The guard refuses to run
# until it's an explicit choice.
from knee.model import DINOV2_BACKBONE

SERIES_TYPES = [SeriesType.SAGITTAL_FLUID, SeriesType.CORONAL_FLUID, SeriesType.AXIAL_FLUID]
BACKBONE = DINOV2_BACKBONE
INPUT_SIZE = 224  # cache/slice resolution AND the ViT's overridden image_size
CROP_MM = 140.0
USE_WEIGHTS = True

WARM_START_FILE: str | None = ...  # FILL from E006: e.g. "e006_sagittal_fluid.pt" (knee-e006-artifacts), or None
if WARM_START_FILE is ...:
    raise RuntimeError("Set WARM_START_FILE from the E006 result (a checkpoint filename or None) before running")

CHECKPOINT_DIR = Path("/kaggle/working")
CACHE_DIR = Path("/tmp/pixel_cache")  # ephemeral: only checkpoints persist as output

In [ ]:
# Fixed 90/10 split — the fine-tune-era regime marker. Same seed always, so every
# fine-tune era experiment shares the split and stays comparable.
import numpy as np

from knee.cv import stratified_holdout
from knee.labels import LABEL_COLUMNS

label_matrix = labels[list(LABEL_COLUMNS)].to_numpy(dtype=np.float32)
val_mask = stratified_holdout(label_matrix, val_fraction=0.1, seed=0)
print(f"split: {int((~val_mask).sum())} train / {int(val_mask.sum())} val")

In [ ]:
# E007: pixel cache at the shared geometry (~75 min decode), then ONE unified
# fine-tune (~45-60 min). The 2 frozen warm-up epochs' printed val scores are the
# frozen baseline on this same split; the per-epoch val macro IS the holdout
# ensemble number (a single model is its own ensemble).
from knee.finetune import FinetuneConfig, build_pixel_cache, finetune_unified
from knee.model import MultiPlaneModel, load_model

cache = build_pixel_cache(
    COMP_ROOT, labels, CACHE_DIR, series_types=SERIES_TYPES,
    input_size=INPUT_SIZE, crop_mm=CROP_MM,
)
print("cache coverage:", {t.value: n for t, n in cache.coverage.items()})

model = MultiPlaneModel(BACKBONE, SERIES_TYPES, image_size=INPUT_SIZE)
if WARM_START_FILE is not None:
    warm = load_model(find_input("knee-e006-artifacts", WARM_START_FILE))
    model.backbone.load_state_dict(warm.model.backbone.state_dict())
    model.head.load_state_dict(warm.model.head.state_dict())
    print(f"warm-started backbone+head from {WARM_START_FILE}")

result = finetune_unified(
    CACHE_DIR, label_matrix, val_mask,
    model=model,
    out_path=CHECKPOINT_DIR / "e007_unified.pt",
    config=FinetuneConfig(),
    input_size=INPUT_SIZE, crop_mm=CROP_MM,
    cell_weights=weights if USE_WEIGHTS else None,
    label_source=BLENDED_LABEL_SOURCE,
)
print(f"E007 unified: best val macro {result.best_val_macro_auc:.3f} (epoch {result.best_epoch + 1})")
print({label: round(auc, 3) for label, auc in result.val_auc_per_label.items()})

In [ ]:
# The single e007_unified.pt persists as notebook output — if shipped, it REPLACES
# all three per-plane checkpoints in knee-weights (inference auto-detects the
# multiplane kind; the plane-prior combiner retires with it). Submission gate
# unchanged: publish/infer only if the holdout macro clears 0.783 decisively AND
# beats E006's paired holdout number.
import math

if run is not None:
    run.config.update({
        "backbone": BACKBONE, "input_size": INPUT_SIZE, "crop_mm": CROP_MM,
        "tier_weighted": USE_WEIGHTS, "model_kind": "multiplane",
        "warm_start": WARM_START_FILE or "none",
    })
    wandb.log({
        "holdout/unified_macro": result.best_val_macro_auc,
        **{f"holdout/unified/{label}": auc for label, auc in result.val_auc_per_label.items() if not math.isnan(auc)},
    })
    run.finish()